# Introduction

In this notebook we are extracting books metadata from Project Gutenberg. Here are following step of execution:

1. Setup and install 2 python libraries [gutenberg](https://pypi.org/project/Gutenberg/) and [request](https://pypi.org/project/requests/)
2. Extract and populate gutenberg metadata with [gutenberg](https://pypi.org/project/Gutenberg/) library
3. Get metadata from each etext number and create a dataframe.
4. Download the latest `pg_catlog.csv` data from gutenberg website
5. Merge and process pg_catog and metadata into a single dataframe.
6. Preprocess and download final metadata.

# Install and Import Library

**gutenberg** library is used for retrieving metadata of the Gutenberg.  This library has been created by [Clemens Wolff](https://justamouse.com/). This library is working based on downloading content from target url `https://www.gutenberg.org/files/{extext_no}/`.

For example: [https://www.gutenberg.org/files/19467](https://www.gutenberg.org/files/19467).

There are all [MIRROR SITE](https://www.gutenberg.org/dirs/MIRRORS.ALL) used by this library too you can check the complete [Documentation](https://pypi.org/project/Gutenberg/) and [Github code](https://github.com/c-w/Gutenberg). Install and setup of **gutenberg** library have been done below.  

In [1]:
!apt install libdb5.3-dev
!pip install gutenberg requests




Suggested packages:
  db5.3-doc
The following NEW packages will be installed:
  libdb5.3-dev
0 upgraded, 1 newly installed, 0 to remove and 129 not upgraded.
Need to get 830 kB of archives.
After this operation, 3,151 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libdb5.3-dev amd64 5.3.28+dfsg1-0.8ubuntu3 [830 kB]
Fetched 830 kB in 0s (8,525 kB/s)

78Selecting previously unselected package libdb5.3-dev.
(Reading database ... 127400 files and directories currently installed.)
Preparing to unpack .../libdb5.3-dev_5.3.28+dfsg1-0.8ubuntu3_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [ 20%] [###########...............................................] 8Unpacking libdb5.3-dev (5.3.28+dfsg1-0.8ubuntu3) ...
7Progress: [ 40%] [#######################...................................] 8Setting up libdb5.3-dev (5.3.28+dfsg1-0.8ubuntu3) ...
7Progress: [ 60%] [###############

In [2]:
from gutenberg.acquire import load_etext, get_metadata_cache
from gutenberg.query import list_supported_metadatas, get_metadata
from tqdm import tqdm
import pandas as pd
import os

# Populate metadata

In [3]:
list_supported_metadatas()

('author', 'formaturi', 'language', 'rights', 'subject', 'title')

In [4]:
cache = get_metadata_cache()
cache.populate()

# Format metadata

`LIMIT` variable is depends this value will increase from future. Hence you can set this by searching this manually
or take help with `pg_catlog`. Take this script below if you things this works

```python
pg_catlog = pd.read_csv("https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv")

LIMIT = pg_catlog.shape[0]
```



In [5]:
def format_metadata(etext_no: int = 1) -> dict:
    """
    Formats and structures metadata for a given Project Gutenberg book.

    This function retrieves metadata for a specified `etext_no` (book identifier) 
    from Project Gutenberg and categorizes different downloadable formats into 
    meaningful labels for better readability.

    Args:
        etext_no (int): The unique identifier for the book in Project Gutenberg. Default is 1.

    Returns:
        dict: A dictionary containing the book's metadata organized with descriptive keys.
    """

    # Retrieve metadata by iterating through all supported metadata fields
    metadata = {
        field: list(get_metadata(field, etext_no))
        for field in list_supported_metadatas()
    }

    # Mapping various formats to their respective labels
    for each in metadata['formaturi']:
        if "html.image" in each:
            metadata['Read online (web)'] = each
        elif "epub3.images" in each:
            metadata['EPUB3 (E-readers incl. Send-to-Kindle)'] = each
        elif "epub.images" in each:
            metadata['EPUB (older E-readers)'] = each
        elif "epub.noimages" in each:
            metadata['EPUB (no images, older E-readers)'] = each
        elif "kf8" in each:
            metadata['Kindle'] = each
        elif "kindle.images" in each:
            metadata['Kindle (E-readers incl. Send-to-Kindle)'] = each
        elif "kindle.noimages" in each:
            metadata['Kindle (no images, older E-readers)'] = each
        elif "txt.utf-8" in each:
            metadata['Plain Text UTF-8'] = each
        elif "h.zip" in each:
            metadata["Download HTML(zip)"] = each
        elif ".rdf" in each:
            metadata["Resource Description Framework (RDF)"] = each
        else:
            # Store unknown or additional links under "Other Links"
            metadata["Other Links"] = metadata.get("Other Links", []) + [each]

    # Remove 'formaturi' field as it's processed and no longer needed
    metadata.pop('formaturi')

    return metadata

In [6]:
# Initialize an empty dictionary to store metadata
# Includes standard fields plus additional downloadable formats
data = {
    field: []  # Initialize each field as an empty list
    for field in [
        "Etext Number"
    ] + list(list_supported_metadatas()) + [
        "Read online (web)", 
        "EPUB3 (E-readers incl. Send-to-Kindle)", 
        "EPUB (older E-readers)",
        "EPUB (no images, older E-readers)", 
        "Kindle", 
        "Kindle (E-readers incl. Send-to-Kindle)", 
        "Kindle (no images, older E-readers)", 
        "Plain Text UTF-8", 
        "Download HTML(zip)",
        "Resource Description Framework (RDF)", 
        "Other Links"
    ]
    if field != "formaturi"  # Exclude 'formaturi' since it's processed separately in `format_metadata`
}

# Limit for the number of entries to process
LIMIT = 75555  

# Iterate through the range of book identifiers with a progress bar
for idx in tqdm(range(1, LIMIT + 1)):

    # Check if the book has a valid title (helps avoid empty or invalid entries)
    if get_metadata('title', idx) != frozenset():
        metadata = format_metadata(idx)  # Fetch metadata for the book
        
        # Fill metadata entries for each field
        for field in data.keys():
            if field == "Etext Number":
                data[field] += [idx]  # Directly assign the book's ID
            else:
                # Fill each metadata field with available data
                data[field] += (
                    [metadata.get(field, "")]  # Add data if it's a string
                    if isinstance(metadata.get(field, ""), str) else
                    [" ; ".join(metadata[field])]  # Join multiple entries with a semicolon
                )

# Convert the collected data into a pandas DataFrame
gutenberg_metadata = pd.DataFrame(data)

# Export the collected data to a CSV file
gutenberg_metadata.to_csv("gutenberg_metadata.csv", index=False)

# Display basic info about the dataset
gutenberg_metadata.info()

# Display the first few rows of the dataset
gutenberg_metadata.head()


100%|██████████| 75555/75555 [02:25<00:00, 518.56it/s]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75379 entries, 0 to 75378
Data columns (total 17 columns):
 #   Column                                   Non-Null Count  Dtype 
---  ------                                   --------------  ----- 
 0   Etext Number                             75379 non-null  int64 
 1   author                                   75379 non-null  object
 2   language                                 75379 non-null  object
 3   rights                                   75379 non-null  object
 4   subject                                  75379 non-null  object
 5   title                                    75379 non-null  object
 6   Read online (web)                        75379 non-null  object
 7   EPUB3 (E-readers incl. Send-to-Kindle)   75379 non-null  object
 8   EPUB (older E-readers)                   75379 non-null  object
 9   EPUB (no images, older E-readers)        75379 non-null  object
 10  Kindle                                   75379 non-null  o

,Etext Number,author,language,rights,subject,title,Read online (web),EPUB3 (E-readers incl. Send-to-Kindle),EPUB (older E-readers),"EPUB (no images, older E-readers)",Kindle,Kindle (E-readers incl. Send-to-Kindle),"Kindle (no images, older E-readers)",Plain Text UTF-8,Download HTML(zip),Resource Description Framework (RDF),Other Links
0,1,"Jefferson, Thomas",en,Public domain in the USA.,E201 ; JK ; United States -- History -- Revolu...,The Declaration of Independence of the United ...,https://www.gutenberg.org/ebooks/1.html.images,https://www.gutenberg.org/ebooks/1.epub3.images,https://www.gutenberg.org/ebooks/1.epub.images,https://www.gutenberg.org/ebooks/1.epub.noimages,https://www.gutenberg.org/ebooks/1.kf8.images,https://www.gutenberg.org/ebooks/1.kindle.images,https://www.gutenberg.org/ebooks/1.kindle.noim...,https://www.gutenberg.org/ebooks/1.txt.utf-8,https://www.gutenberg.org/files/1/1-h.zip,https://www.gutenberg.org/ebooks/1.rdf,https://www.gutenberg.org/files/1/1-h/1-h.htm ...
1,2,United States,en,Public domain in the USA.,Civil rights -- United States -- Sources ; JK ...,The United States Bill of Rights\r\nThe Ten Or...,https://www.gutenberg.org/ebooks/2.html.images,https://www.gutenberg.org/ebooks/2.epub3.images,https://www.gutenberg.org/ebooks/2.epub.images,https://www.gutenberg.org/ebooks/2.epub.noimages,https://www.gutenberg.org/ebooks/2.kf8.images,https://www.gutenberg.org/ebooks/2.kindle.images,https://www.gutenberg.org/ebooks/2.kindle.noim...,https://www.gutenberg.org/ebooks/2.txt.utf-8,https://www.gutenberg.org/files/2/2-h.zip,https://www.gutenberg.org/ebooks/2.rdf,https://www.gutenberg.org/files/2/2-h/2-h.htm ...
2,3,"Kennedy, John F. (John Fitzgerald)",en,Public domain in the USA.,E838 ; United States -- Foreign relations -- 1...,John F. Kennedy's Inaugural Address,https://www.gutenberg.org/ebooks/3.html.images,https://www.gutenberg.org/ebooks/3.epub3.images,https://www.gutenberg.org/ebooks/3.epub.images,https://www.gutenberg.org/ebooks/3.epub.noimages,https://www.gutenberg.org/ebooks/3.kf8.images,https://www.gutenberg.org/ebooks/3.kindle.images,https://www.gutenberg.org/ebooks/3.kindle.noim...,https://www.gutenberg.org/ebooks/3.txt.utf-8,https://www.gutenberg.org/cache/epub/3/pg3-h.zip,https://www.gutenberg.org/ebooks/3.rdf,https://www.gutenberg.org/cache/epub/3/pg3.cov...
3,4,"Lincoln, Abraham",en,Public domain in the USA.,E456 ; Soldiers' National Cemetery (Gettysburg...,Lincoln's Gettysburg Address\r\nGiven November...,https://www.gutenberg.org/ebooks/4.html.images,https://www.gutenberg.org/ebooks/4.epub3.images,https://www.gutenberg.org/ebooks/4.epub.images,https://www.gutenberg.org/ebooks/4.epub.noimages,https://www.gutenberg.org/ebooks/4.kf8.images,https://www.gutenberg.org/ebooks/4.kindle.images,https://www.gutenberg.org/ebooks/4.kindle.noim...,https://www.gutenberg.org/ebooks/4.txt.utf-8,https://www.gutenberg.org/cache/epub/4/pg4-h.zip,https://www.gutenberg.org/ebooks/4.rdf,https://www.gutenberg.org/files/4/4.zip ; http...
4,5,United States,en,Public domain in the USA.,JK ; United States. Constitution ; United Stat...,The United States Constitution,https://www.gutenberg.org/ebooks/5.html.images,https://www.gutenberg.org/ebooks/5.epub3.images,https://www.gutenberg.org/ebooks/5.epub.images,https://www.gutenberg.org/ebooks/5.epub.noimages,https://www.gutenberg.org/ebooks/5.kf8.images,https://www.gutenberg.org/ebooks/5.kindle.images,https://www.gutenberg.org/ebooks/5.kindle.noim...,https://www.gutenberg.org/ebooks/5.txt.utf-8,https://www.gutenberg.org/files/5/5-h.zip,https://www.gutenberg.org/ebooks/5.rdf,https://www.gutenberg.org/files/5/5.txt ; http...


# Download and Merging pg_catlog.csv

`pg_catlog.csv` is update update in gutenberg almost the same like metadata without info about **URLS, rights** This will also help above for setting `LIMIT` variable. Considering this will be beneficial for enriching quality. Hence this file is consider.

In [7]:
# Load the Project Gutenberg catalog data directly from the provided URL
pg_catlog = pd.read_csv("https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv")

# Display a concise summary of the dataset, including data types and non-null values
pg_catlog.info()

# Display the first five rows of the dataset for a quick preview
pg_catlog.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75394 entries, 0 to 75393
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Text#        75394 non-null  int64 
 1   Type         75394 non-null  object
 2   Issued       75394 non-null  object
 3   Title        75394 non-null  object
 4   Language     75394 non-null  object
 5   Authors      75226 non-null  object
 6   Subjects     75333 non-null  object
 7   LoCC         75126 non-null  object
 8   Bookshelves  75046 non-null  object
dtypes: int64(1), object(8)
memory usage: 5.2+ MB


,Text#,Type,Issued,Title,Language,Authors,Subjects,LoCC,Bookshelves
0,1,Text,1971-12-01,The Declaration of Independence of the United ...,en,"Jefferson, Thomas, 1743-1826","United States -- History -- Revolution, 1775-1...",E201; JK,Politics; American Revolutionary War; United S...
1,2,Text,1972-12-01,The United States Bill of Rights\r\nThe Ten Or...,en,United States,Civil rights -- United States -- Sources; Unit...,JK; KF,Politics; American Revolutionary War; United S...
2,3,Text,1973-11-01,John F. Kennedy's Inaugural Address,en,"Kennedy, John F. (John Fitzgerald), 1917-1963",United States -- Foreign relations -- 1961-196...,E838,Browsing: History - American; Browsing: Politics
3,4,Text,1973-11-01,Lincoln's Gettysburg Address\r\nGiven November...,en,"Lincoln, Abraham, 1809-1865",Consecration of cemeteries -- Pennsylvania -- ...,E456,US Civil War; Browsing: History - American; Br...
4,5,Text,1975-12-01,The United States Constitution,en,United States,United States -- Politics and government -- 17...,JK; KF,United States; Politics; American Revolutionar...


In [8]:
# Rename the 'Text#' column to 'Etext Number' for consistency across datasets
pg_catlog.rename(columns={"Text#": "Etext Number"}, inplace=True)

# Merge 'pg_catlog' with 'gutenberg_metadata' on the 'Etext Number' column
# 'how="left"' ensures that all rows from 'pg_catlog' are retained,
# and matching data from 'gutenberg_metadata' is merged where available
gutenberg_metadata = pg_catlog.merge(gutenberg_metadata, on="Etext Number", how='left')

# Preprocessing ( Removing Duplicate columns from merged csv)

Some columns like *Author, Subject, Title, Language* were repeated hence combine them into single final column. Below function handle such cases

In [9]:
def merge_preprocessor(feature_1: str, feature2: str):
    """
    Merges two specified columns in the 'gutenberg_metadata' DataFrame 
    by filling missing values in one column using data from the other.

    Parameters:
    -----------
    feature_1 : str
        The first column name for merging.
    feature2 : str
        The second column name for merging.

    Process:
    --------
    - Identifies which column has fewer missing values.
    - Fills missing values in the column with more null values using the one with fewer null values.
    - Ensures data consistency by updating the DataFrame accordingly.
    """

    # Inner function to handle merging logic
    def merge(f1, f2, extra=False):
        """
        Handles the core merging logic.

        Parameters:
        -----------
        f1 : str
            The column that needs missing values filled.
        f2 : str
            The column providing values to fill missing entries in 'f1'.
        extra : bool, optional
            If True, ensures additional non-null entries are updated for consistency.
        """
        # Identify rows where 'f1' is null and 'f2' is not null
        data = gutenberg_metadata[
            gutenberg_metadata[f1].isna() &  # Missing values in column 'f1'
            ~gutenberg_metadata[f2].isna()   # Non-missing values in column 'f2'
        ]
        # Fill missing values in 'f1' with values from 'f2'
        data[f1] = data[f2]

        # Update the main DataFrame with these corrected values
        gutenberg_metadata.update(data)

        # Additional merging logic if 'extra=True'
        if extra:
            data = gutenberg_metadata[[feature_1, feature2]].dropna()  # Filter non-null rows
            data[feature_1] = data[feature2]  # Copy 'feature2' values into 'feature_1'
            gutenberg_metadata.update(data)  # Update the DataFrame

    # Identify the column with fewer null values
    min_null_feature = (
        gutenberg_metadata[[feature_1, feature2]]
        .isna().sum()     # Count null values
        .sort_values()    # Sort by ascending null count
        .head(1)          # Get the column with the fewest nulls
        .index[0]         # Extract column name
    )

    # Perform merging based on the column with fewer null values
    if min_null_feature == feature_1:
        # If 'feature_1' has fewer nulls, fill 'feature_1' using 'feature2'
        merge(feature_1, feature2)
        # Drop 'feature2' as its data has been transferred
        gutenberg_metadata.drop(columns=[feature2], inplace=True)
    else:
        # If 'feature2' has fewer nulls, fill 'feature2' using 'feature_1'
        merge(feature2, feature_1, True)
        # Drop 'feature_1' and rename 'feature2' as 'feature_1'
        gutenberg_metadata.drop(columns=[feature_1], inplace=True)
        gutenberg_metadata.rename(columns={feature2: feature_1}, inplace=True)


In [10]:
# Dictionary mapping primary feature names to their alternative column names
preprocess_feature = {
    "Authors": "author",        # Map 'Authors' to 'author'
    "Subjects": "subject",      # Map 'Subjects' to 'subject'
    "Title": "title",           # Map 'Title' to 'title'
    "Language": "language"      # Map 'Language' to 'language'
}

# Iterate through each feature in the dictionary
for feature in preprocess_feature.keys():
    # Call the merge_preprocessor function to merge data between
    # the specified feature and its corresponding alternative column
    merge_preprocessor(feature, preprocess_feature[feature])


<ipython-input-9-14329665cedb>:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[f1] = data[f2]
<ipython-input-9-14329665cedb>:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[f1] = data[f2]


# Saving Final DataFrame

In [11]:
gutenberg_metadata.to_csv("gutenberg_metadata.csv", index=False)
gutenberg_metadata.info()
gutenberg_metadata.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75394 entries, 0 to 75393
Data columns (total 21 columns):
 #   Column                                   Non-Null Count  Dtype 
---  ------                                   --------------  ----- 
 0   Etext Number                             75394 non-null  int64 
 1   Type                                     75394 non-null  object
 2   Issued                                   75394 non-null  object
 3   Title                                    75394 non-null  object
 4   Language                                 75394 non-null  object
 5   LoCC                                     75126 non-null  object
 6   Bookshelves                              75046 non-null  object
 7   Authors                                  75391 non-null  object
 8   rights                                   75377 non-null  object
 9   Subjects                                 75380 non-null  object
 10  Read online (web)                        75377 non-null  o

,Etext Number,Type,Issued,Title,Language,LoCC,Bookshelves,Authors,rights,Subjects,...,EPUB3 (E-readers incl. Send-to-Kindle),EPUB (older E-readers),"EPUB (no images, older E-readers)",Kindle,Kindle (E-readers incl. Send-to-Kindle),"Kindle (no images, older E-readers)",Plain Text UTF-8,Download HTML(zip),Resource Description Framework (RDF),Other Links
0,1,Text,1971-12-01,The Declaration of Independence of the United ...,en,E201; JK,Politics; American Revolutionary War; United S...,"Jefferson, Thomas",Public domain in the USA.,E201 ; JK ; United States -- History -- Revolu...,...,https://www.gutenberg.org/ebooks/1.epub3.images,https://www.gutenberg.org/ebooks/1.epub.images,https://www.gutenberg.org/ebooks/1.epub.noimages,https://www.gutenberg.org/ebooks/1.kf8.images,https://www.gutenberg.org/ebooks/1.kindle.images,https://www.gutenberg.org/ebooks/1.kindle.noim...,https://www.gutenberg.org/ebooks/1.txt.utf-8,https://www.gutenberg.org/files/1/1-h.zip,https://www.gutenberg.org/ebooks/1.rdf,https://www.gutenberg.org/files/1/1-h/1-h.htm ...
1,2,Text,1972-12-01,The United States Bill of Rights\r\nThe Ten Or...,en,JK; KF,Politics; American Revolutionary War; United S...,United States,Public domain in the USA.,Civil rights -- United States -- Sources ; JK ...,...,https://www.gutenberg.org/ebooks/2.epub3.images,https://www.gutenberg.org/ebooks/2.epub.images,https://www.gutenberg.org/ebooks/2.epub.noimages,https://www.gutenberg.org/ebooks/2.kf8.images,https://www.gutenberg.org/ebooks/2.kindle.images,https://www.gutenberg.org/ebooks/2.kindle.noim...,https://www.gutenberg.org/ebooks/2.txt.utf-8,https://www.gutenberg.org/files/2/2-h.zip,https://www.gutenberg.org/ebooks/2.rdf,https://www.gutenberg.org/files/2/2-h/2-h.htm ...
2,3,Text,1973-11-01,John F. Kennedy's Inaugural Address,en,E838,Browsing: History - American; Browsing: Politics,"Kennedy, John F. (John Fitzgerald)",Public domain in the USA.,E838 ; United States -- Foreign relations -- 1...,...,https://www.gutenberg.org/ebooks/3.epub3.images,https://www.gutenberg.org/ebooks/3.epub.images,https://www.gutenberg.org/ebooks/3.epub.noimages,https://www.gutenberg.org/ebooks/3.kf8.images,https://www.gutenberg.org/ebooks/3.kindle.images,https://www.gutenberg.org/ebooks/3.kindle.noim...,https://www.gutenberg.org/ebooks/3.txt.utf-8,https://www.gutenberg.org/cache/epub/3/pg3-h.zip,https://www.gutenberg.org/ebooks/3.rdf,https://www.gutenberg.org/cache/epub/3/pg3.cov...
3,4,Text,1973-11-01,Lincoln's Gettysburg Address\r\nGiven November...,en,E456,US Civil War; Browsing: History - American; Br...,"Lincoln, Abraham",Public domain in the USA.,E456 ; Soldiers' National Cemetery (Gettysburg...,...,https://www.gutenberg.org/ebooks/4.epub3.images,https://www.gutenberg.org/ebooks/4.epub.images,https://www.gutenberg.org/ebooks/4.epub.noimages,https://www.gutenberg.org/ebooks/4.kf8.images,https://www.gutenberg.org/ebooks/4.kindle.images,https://www.gutenberg.org/ebooks/4.kindle.noim...,https://www.gutenberg.org/ebooks/4.txt.utf-8,https://www.gutenberg.org/cache/epub/4/pg4-h.zip,https://www.gutenberg.org/ebooks/4.rdf,https://www.gutenberg.org/files/4/4.zip ; http...
4,5,Text,1975-12-01,The United States Constitution,en,JK; KF,United States; Politics; American Revolutionar...,United States,Public domain in the USA.,JK ; United States. Constitution ; United Stat...,...,https://www.gutenberg.org/ebooks/5.epub3.images,https://www.gutenberg.org/ebooks/5.epub.images,https://www.gutenberg.org/ebooks/5.epub.noimages,https://www.gutenberg.org/ebooks/5.kf8.images,https://www.gutenberg.org/ebooks/5.kindle.images,https://www.gutenberg.org/ebooks/5.kindle.noim...,https://www.gutenberg.org/ebooks/5.txt.utf-8,https://www.gutenberg.org/files/5/5-h.zip,https://www.gutenberg.org/ebooks/5.rdf,https://www.gutenberg.org/files/5/5.txt ; http...


In [12]:
gutenberg_metadata.isna().sum()

Etext Number                                 0
Type                                         0
Issued                                       0
Title                                        0
Language                                     0
LoCC                                       268
Bookshelves                                348
Authors                                      3
rights                                      17
Subjects                                    14
Read online (web)                           17
EPUB3 (E-readers incl. Send-to-Kindle)      17
EPUB (older E-readers)                      17
EPUB (no images, older E-readers)           17
Kindle                                      17
Kindle (E-readers incl. Send-to-Kindle)     17
Kindle (no images, older E-readers)         17
Plain Text UTF-8                            17
Download HTML(zip)                          17
Resource Description Framework (RDF)        17
Other Links                                 17
dtype: int64